# Tipos de Datos y Datasets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelSkrauba/applied-ai-engineering/blob/main/notebooks/01_introduccion/04_tipos_de_datos_y_datasets.ipynb)



## Objetivos
- Diferenciar entre datos estructurados, semiestructurados y no estructurados.
- Entender los formatos de almacenamiento más comunes en ingeniería de datos (CSV, Parquet, SQL).
- Conocer los principales repositorios para obtener datasets de calidad.

## Prerrequisitos
- [Ciclo de Vida de un Proyecto de ML](02_ciclo_de_vida_ml.ipynb).

---
## Configuración del Entorno

In [1]:
# @title *Esta celda clona el repositorio (en Colab) e importa las utilidades comunes*
import sys
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    REPO_NAME = "applied-ai-engineering"
    if not os.path.exists(REPO_NAME):
        subprocess.run(["git", "clone", "https://github.com/AxelSkrauba/applied-ai-engineering.git"], check=True)
    os.chdir(f"/content/{REPO_NAME}")
    sys.path.append(f"/content/{REPO_NAME}")

from utils.plots import setup_plot_style
setup_plot_style()

In [6]:
import pandas as pd
import numpy as np
import os
import sqlite3
import warnings
warnings.filterwarnings('ignore')

## 1. La Naturaleza de los Datos



En Machine Learning, el tipo de algoritmo que se elija dependerá fundamentalmente de la estructura de los datos. Podemos dividirlos en tres grandes categorías:

### 1.1 Datos Estructurados


Están altamente organizados y formateados de manera que sea fácil buscar en ellos en bases de datos relacionales. Podemos pensar en **filas y columnas**, dónde generalmente, las columnas hacen referencia a variables o *features*, y las filas son las observaciones o muestras.
- **Ejemplos:** Tablas de Excel, bases de datos SQL, archivos CSV (del inglés *Comma Separated Values*, aunque por definición utiliza comas como separador, en la práctica muchos archivos usan otros delimitadores como punto y coma (`;`), tabulaciones (`\t`) o espacios.).
- **Ventajas:** Fáciles de procesar, analizar y alimentar a algoritmos clásicos de ML (Regresión Lineal, Random Forest...), ya tienen la estructura esperada por los mismos.

### 1.2 Datos No Estructurados


No tienen un modelo de datos predefinido o no están organizados de manera predefinida. Representan la mayor parte de los datos generados hoy en día.
- **Ejemplos:** Imágenes, archivos de audio, texto libre (tweets, libros), videos.
- **Cómo se procesan:** Requieren técnicas de Deep Learning (Redes Neuronales Convolucionales para imágenes, Modelos de Lenguaje como Transformers para texto) para extraer "características" estructuradas antes de poder hacer predicciones.

En muchos *pipelines* modernos, estos datos primero se transforman en representaciones numéricas (*embeddings*, vectores de características) antes de alimentar modelos. Esto se entiende como "estructurar los datos", es decir, obtener los *features*. Lo vemos cuando amerite...

### 1.3 Datos Semi-estructurados


Contienen etiquetas u otros marcadores para separar elementos semánticos, pero no tienen una estructura tabular estricta.
- **Ejemplos:** Archivos JSON (provenientes de APIs, logs de sistemas), XML, correos electrónicos (tienen un "Para", "De", pero el cuerpo es texto libre).

## 2. Formatos de Almacenamiento Comunes y Acceso a Datos



A medida que los datos crecen, la forma en que los guardamos importa (y mucho).

*   **CSV (Comma Separated Values):** El más universal. Es texto plano. Lento para leer si es muy grande y no preserva los tipos de datos (un número puede ser interpretado como string). Muchas librerías (`pandas`) infieren el tipo de dato automáticamente, basados en el contexto.
*   **Parquet:** Formato de almacenamiento columnar y binario ampliamente utilizado en ecosistemas de Big Data (Spark, Hadoop, DuckDB, etc.) Está altamente comprimido, lee rapidísimo y conserva la estructura de los datos (sabe qué columna es `int` y cuál `datetime`). En general, se prefiere Parquet cuando los datasets empiezan a crecer (decenas o cientos de MB o más) o cuando se realizan lecturas analíticas frecuentes.
*   **SQL (Bases de Datos Relacionales):** Los datos viven en servidores. Extraemos solo lo que necesitamos usando consultas (*queries*). Es vital para un ingeniero conocer al menos lo básico de SQL.

### Breve demostración: Leyendo desde SQL vs CSV


Imaginemos que tenemos una base de datos local SQLite con registros de sensores.

In [18]:
# 1. Creamos una base de datos ficticia en memoria (o en archivo)
conn = sqlite3.connect(':memory:')

# Creamos algunos datos en un DataFrame de pandas
df_sensores = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-01', periods=5, freq='h'),
    'sensor_id': ['S1', 'S2', 'S1', 'S3', 'S2'],
    'temperatura': [22.5, 23.1, 22.8, 19.5, 23.4]
})

# Guardamos el DataFrame en la base de datos SQL
df_sensores.to_sql('lecturas_sensores', conn, index=False)

# 2. Consultamos usando lenguaje SQL
query = """
SELECT sensor_id, AVG(temperatura) as temp_promedio
FROM lecturas_sensores
GROUP BY sensor_id
ORDER BY temp_promedio DESC;
"""

# Pandas puede ejecutar la query y devolver un DataFrame directamente
resultado = pd.read_sql_query(query, conn)
print("Resultados de la consulta SQL:")
print(resultado)

conn.close()

Resultados de la consulta SQL:
  sensor_id  temp_promedio
0        S2          23.25
1        S1          22.65
2        S3          19.50


In [19]:
# El DataFrame completo sería:
df_sensores

,timestamp,sensor_id,temperatura
0,2024-01-01 00:00:00,S1,22.5
1,2024-01-01 01:00:00,S2,23.1
2,2024-01-01 02:00:00,S1,22.8
3,2024-01-01 03:00:00,S3,19.5
4,2024-01-01 04:00:00,S2,23.4


*Nota: Aunque Pandas puede hacer estas agrupaciones fácilmente en memoria, si la tabla tiene 50 millones de filas, no cabrá en la RAM de la computadora. Ahí es donde SQL brilla: el servidor de base de datos hace el trabajo pesado y solo envía lo solicitado.*

In [5]:
# Pequeña comparativa de los tamaños al volcar a disco los archivos

df_sensores.to_csv("sensores.csv", index=False)   # Texto plano
df_sensores.to_parquet("sensores.parquet")        # Binario

print("CSV size:", os.path.getsize("sensores.csv"))
print("Parquet size:", os.path.getsize("sensores.parquet"))

CSV size: 172
Parquet size: 2786


A primera vista puede parecer extraño que el archivo Parquet sea más grande que el CSV.
Esto ocurre porque nuestro dataset es extremadamente pequeño.

El formato Parquet almacena información adicional (*metadata*, esquema de columnas, compresión y optimización para lectura columnar). En datasets muy pequeños este *overhead* hace que el archivo ocupe más espacio que un CSV simple.

Sin embargo, cuando el volumen de datos crece, Parquet suele ser mucho más eficiente en tamaño y velocidad de lectura, especialmente para análisis de datos.

Para ilustrar esto, generamos un dataset sintético más grande y comparamos nuevamente los tamaños.

In [7]:
# Generamos un dataset más grande
n = 200000

df_grande = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01", periods=n, freq="min"),
    "sensor_id": np.random.choice(["S1","S2","S3","S4","S5"], n),
    "temperatura": np.random.normal(22, 2, n).round(2),
    "humedad": np.random.normal(60, 10, n).round(2)
})

# Guardamos en ambos formatos
df_grande.to_csv("sensores_grande.csv", index=False)
df_grande.to_parquet("sensores_grande.parquet")

# Comparamos tamaños
size_csv = os.path.getsize("sensores_grande.csv")
size_parquet = os.path.getsize("sensores_grande.parquet")

print("CSV size:", size_csv)
print("Parquet size:", size_parquet)

print("\nRatio CSV/Parquet:", round(size_csv/size_parquet,2))

CSV size: 6959924
Parquet size: 2396012

Ratio CSV/Parquet: 2.9


Ahora podemos observar una diferencia mucho más clara entre ambos formatos. Casi un ratio de 3 para este caso simple.

Y ya que estamos, vemos una pequeña comparativa para los tiempos de lectura:

In [16]:
import time

# Tiempo lectura CSV
start = time.time()
pd.read_csv("sensores_grande.csv")
csv_time = time.time() - start

# Tiempo lectura Parquet
start = time.time()
pd.read_parquet("sensores_grande.parquet")
parquet_time = time.time() - start

print("Tiempo lectura CSV:", round(csv_time,3), "segundos")
print("Tiempo lectura Parquet:", round(parquet_time,3), "segundos")

Tiempo lectura CSV: 0.184 segundos
Tiempo lectura Parquet: 0.021 segundos


Ni hablar de qué pasa en datasets realmente grandes, Parquet permite leer únicamente las columnas necesarias, lo que acelera mucho los análisis y optimiza el uso de recursos (se carga lo que se necesita).

Por lo general, vamos a usar igualmente archivos `.csv` para datos estructurados. Por su practicidad y facilidad de abrir e incluso editar con literalmente cualquier cosa (es texto plano). Pero sepan que existen formatos especializados.

## 3. ¿Dónde encontrar Datasets?



La práctica hace al maestro. Van algunos de los mejores lugares para encontrar datos reales y ensuciarte las manos:

1.  **[Kaggle](https://www.kaggle.com/datasets):** La meca de los Data Scientists. Miles de datasets limpios y competencias (incluso con premios en dinero). Se encuentran notebooks con soluciones de la comunidad, incluso los ganadores de competencias suelen disponibilizar su flujo de trabajo, técnicas utilizadas, pipelines que si funcionaron, los que no, etc.
2.  **[UCI Machine Learning Repository](https://archive.ics.uci.edu/):** Uno de los repositorios más antiguos. Datasets muy clásicos (ideales para benchmarks académicos).
3.  **[Hugging Face Datasets](https://huggingface.co/datasets):** El mejor lugar para encontrar datos no estructurados (especialmente texto para NLP y audio).
4.  **Bancos de Datos Gubernamentales:** Prácticamente todos los países y organizaciones (como el [Banco Mundial](https://data.worldbank.org/)) tienen portales de datos abiertos. Muy útiles para proyectos con impacto social.

## Resultados y Discusión


En el mundo real, rara vez se recibe un archivo CSV limpio y listo para usar. Es importante estar preparado para conectarte a bases de datos (SQL, PostgreSQL, MongoDB), consumir APIs, extraer datos de páginas web (Web Scraping) o directamente establecer los protocolos para la generación y obtención de los datos que se necesitan para un *pipeline* (implica responder preguntas como: ¿Cuántos datos necesito realmente? ¿Puedo obtener los que necesito o el costo es prohibitivo? ¿Aspectos éticos involucrados? ¿Cómo garantizo la representatividad? ¿Posibles sesgos?... la lista puede crecer mucho).





## Conexiones y Próximos Pasos
- ➡️ **Siguiente:** Antes de descargar terabytes de datos, necesitamos preparar nuestra máquina (**OPCIONAL**, pero recomendado). Vamos a [Configuración del Entorno Local](05_configuracion_entorno_local.ipynb).

---
## Entorno de Ejecución

In [9]:
from utils.environment import environment_table
environment_table()

Package,Version
Python,3.12.12
Platform,Linux-6.6.113+-x86_64-with-glibc2.35
IPython,7.34.0
ipywidgets,7.7.1
matplotlib,3.10.0
numpy,2.0.2
pandas,2.2.2
scipy,1.16.3
seaborn,0.13.2
statsmodels,0.14.6
